# Turnkey runner — train & compare all three models

One-click pipeline on **Google Colab (GPU)**: download → preprocess →
train *image / text / fusion* → evaluate each on test → assemble the
**3-way comparison** (the project's headline result).

Expected wall-clock on a free Colab GPU: roughly 20–40 min total for the
~20k-row subset (image/fusion dominate; the EfficientNet backbone is frozen).

## 0. Setup

In [ ]:
# On Colab: clone your repo, then run from its root (uncomment + edit).
# !git clone <YOUR_REPO_URL> repo
# %cd repo
!pip -q install -r requirements.txt
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
from src import config
config.set_seeds()
print('seed', config.SEED, '| epochs', config.EPOCHS, '| batch', config.BATCH_SIZE)

## 1. Data: download + stratified splits

In [ ]:
from src.data.download import download
from src.data.preprocess import preprocess
download()
preprocess()

In [ ]:
# Build + persist the shared TextVectorization (adapted on TRAIN only).
from src.data.dataset import get_text_vectorizer
vec = get_text_vectorizer()
print('vocab size:', len(vec.get_vocabulary()))

## 2. Train the three models
Each call saves weights + metrics + training curves under `artifacts/<model>/`.

In [ ]:
from src.train import train
train('image')

In [ ]:
train('text')

In [ ]:
train('fusion')

## 3. Evaluate each on the held-out test set
Writes `test_metrics.json` + a confusion matrix per model.

In [ ]:
from src.evaluate import evaluate
for m in ('image', 'text', 'fusion'):
    evaluate(m)

## 4. The 3-way comparison (headline result)

In [ ]:
from src.compare import compare
df = compare()
df

In [ ]:
# Show the saved comparison chart and confusion matrices inline.
from IPython.display import Image as IPImage, display
display(IPImage(str(config.ARTIFACTS_DIR / 'comparison.png')))
for m in ('image', 'text', 'fusion'):
    print(m)
    display(IPImage(str(config.ARTIFACTS_DIR / m / 'confusion_matrix.png')))

## 5. (Optional, Phase 4) Lightweight hyperparameter search on fusion
A handful of manual runs from `config.HP_GRID` — no AutoML.

In [ ]:
import json
from src.evaluate import evaluate
results = []
for i, hp in enumerate(config.HP_GRID):
    name = f"fusion_hp{i}"
    s = train('fusion', learning_rate=hp['learning_rate'], dropout=hp['dropout'],
              fusion_units=hp['fusion_dense_units'], run_name=name)
    results.append((name, hp, s['best_val_accuracy']))
for name, hp, val_acc in sorted(results, key=lambda r: -r[2]):
    print(f'{name}: val_acc={val_acc:.4f}  {hp}')

## 6. (Optional stretch) Stage-2 fine-tuning
Unfreeze the EfficientNet backbone and train at a low LR. Keep only if it helps.

In [ ]:
# s = train('fusion', learning_rate=1e-5, trainable_backbone=True, run_name='fusion_ft')
# evaluate('fusion', run_name='fusion_ft')